# `encode_vector` — Vector Input with Auto-Detection

Pass a NumPy array directly. PyEncode auto-detects the pattern or verifies a declared type.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
from qiskit.quantum_info import Statevector
from pyencode import encode_vector, VectorType

## Auto-detection (no type hint)

PyEncode tries each extractor in order: DISCRETE, UNIFORM, STEP, SQUARE, SINE, COSINE, MULTI_DISCRETE, MULTI_SINE.

In [ ]:
# DISCRETE: single nonzero entry
f = np.zeros(8); f[3] = 5.0
circuit, info = encode_vector(f)
print(info)
circuit.draw('mpl')

In [ ]:
# UNIFORM: all entries equal
f = np.full(8, 3.0)
circuit, info = encode_vector(f)
print(info)
circuit.draw('mpl')

In [ ]:
# STEP: prefix constant, rest zero
f = np.zeros(8); f[:4] = 2.0
circuit, info = encode_vector(f)
print(info)
circuit.draw('mpl')

In [ ]:
# SQUARE: contiguous block
f = np.zeros(16); f[4:12] = 1.0
circuit, info = encode_vector(f)
print(info)
circuit.draw('mpl')

In [ ]:
# SINE: pure sinusoid
N = 16; k = np.arange(N)
f = np.sin(2 * np.pi * 1 * k / N)
circuit, info = encode_vector(f)
print(info)
circuit.draw('mpl')

In [ ]:
# COSINE: pure cosine
N = 16; k = np.arange(N)
f = np.cos(2 * np.pi * 1 * k / N)
circuit, info = encode_vector(f)
print(info)
circuit.draw('mpl')

In [ ]:
# MULTI_DISCRETE: multiple point loads
f = np.zeros(8); f[1] = 2.0; f[5] = 3.0; f[7] = 1.0
circuit, info = encode_vector(f)
print(info)
circuit.draw('mpl')

In [ ]:
# MULTI_SINE: sum of two sinusoidal modes
N = 16; k = np.arange(N)
f = 2.0 * np.sin(2*np.pi*1*k/N) + np.sin(2*np.pi*3*k/N)
circuit, info = encode_vector(f)
print(info)

## With declared type (verified mode)

When `vector_type` is provided, PyEncode extracts parameters for that type only and verifies the match. Raises an error if the vector doesn't match.

In [ ]:
# Correct declaration
f = np.zeros(8); f[3] = 5.0
circuit, info = encode_vector(f, vector_type="DISCRETE")
print(f"Declared DISCRETE -> {info.vector_type}, params={info.params}")

In [ ]:
# Wrong declaration raises informative error
f = np.full(8, 3.0)
try:
    encode_vector(f, vector_type="DISCRETE")
except ValueError as e:
    print(f"Error: {e}")

In [ ]:
# Sine with verification
N = 16; k = np.arange(N)
f = np.sin(2 * np.pi * 3 * k / N)
circuit, info = encode_vector(f, vector_type="SINE")
print(f"Verified SINE: n={info.params['n']}, phi={info.params['phi']}")

## Preference order: simpler patterns win

In [ ]:
# Point load at k=0: could be DISCRETE or STEP(k_s=1)
# Auto-detect prefers DISCRETE (simpler)
f = np.zeros(8); f[0] = 1.0
_, info = encode_vector(f)
print(f"Detected as: {info.vector_type}  (DISCRETE preferred over STEP)")

## Unrecognised vectors

In [ ]:
# Random vector: no pattern matches
rng = np.random.default_rng(42)
f = rng.standard_normal(8)
try:
    encode_vector(f)
except ValueError as e:
    print(f"Error: {e}")